# Diamond Pipeline — IGI PDF Extraction

### The problem
Cloudflare on `api.igi.org` blocks **all Google Cloud IPs** (including Colab).
No TLS trick (curl_cffi, cloudscraper, Selenium) will work — the IP itself is blocked.

### The solution
Run `download_igi_pdfs.py` on your **local machine** (your home IP isn't blocked):

```bash
pip install requests pdfplumber pandas
python download_igi_pdfs.py
```

Then upload `diamonds_full.csv` here and continue.

### This notebook
1. Loads the output CSV from the local script
2. Shows extraction results & stats
3. Filters diamonds by ideal proportions

In [ ]:
import pandas as pd
import os

# Find the CSV
csv_path = None
for p in ["/content/diamonds_full.csv", "diamonds_full.csv"]:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    print("diamonds_full.csv not found!")
    print("")
    print("Run this on your local machine first:")
    print("  1. Download: download_igi_pdfs.py + luvansh_updated.csv")
    print("  2. pip install requests pdfplumber pandas")
    print("  3. python download_igi_pdfs.py")
    print("  4. Upload diamonds_full.csv to this Colab")
else:
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} diamonds from {csv_path}")

    # Stats
    total = len(df)
    has_crown = df["pdf_crown_angle"].notna().sum()
    has_error = df["pdf_error"].notna().sum()
    print(f"\nPDF data extracted: {has_crown}/{total}")
    print(f"Errors: {has_error}/{total}")

    # Column summary
    print(f"\n{'Column':<25} {'Non-null':>8}")
    print("-" * 40)
    for col in df.columns:
        nn = df[col].notna().sum()
        print(f"{col:<25} {nn:>8}")

    df.head(5)

## Filter by Ideal Proportions

In [ ]:
IDEAL_RANGES = {
    "pdf_lw_ratio":       (1.00, 1.02),
    "pdf_table_pct":      (54.0, 58.0),
    "pdf_depth_pct":      (61.0, 62.3),
    "pdf_crown_angle":    (34.0, 35.0),
    "pdf_pavilion_angle": (40.6, 40.9),
    "pdf_crown_height":   (14.0, 16.0),
    "pdf_pavilion_depth": (42.5, 43.2),
}

if 'df' in dir() and "pdf_crown_angle" in df.columns:
    # Only check diamonds that have PDF data
    has_data = df["pdf_crown_angle"].notna()
    df_check = df[has_data].copy()

    if len(df_check) == 0:
        print("No diamonds have PDF data yet — run the local script first.")
    else:
        # Check each criterion
        df_check["criteria_met"] = 0
        for col, (lo, hi) in IDEAL_RANGES.items():
            if col in df_check.columns:
                in_range = df_check[col].between(lo, hi)
                df_check["criteria_met"] += in_range.astype(int)

        n_criteria = len(IDEAL_RANGES)
        df_check["ideal"] = df_check["criteria_met"] == n_criteria

        ideal = df_check[df_check["ideal"]]
        near = df_check[df_check["criteria_met"] >= n_criteria - 1]

        print(f"Checked {len(df_check)} diamonds with PDF data")
        print(f"IDEAL (all {n_criteria} criteria): {len(ideal)}")
        print(f"NEAR-IDEAL ({n_criteria-1}+ criteria): {len(near)}")

        if len(ideal) > 0:
            show_cols = ["web_sku", "carat", "color", "clarity", "price",
                         "pdf_table_pct", "pdf_depth_pct",
                         "pdf_crown_angle", "pdf_pavilion_angle",
                         "pdf_crown_height", "pdf_pavilion_depth",
                         "pdf_lw_ratio", "product_url"]
            show_cols = [c for c in show_cols if c in ideal.columns]
            display(ideal[show_cols])
        elif len(near) > 0:
            print("\nNearest matches:")
            show_cols = ["web_sku", "carat", "price", "criteria_met",
                         "pdf_crown_angle", "pdf_pavilion_angle",
                         "pdf_crown_height", "pdf_pavilion_depth"]
            show_cols = [c for c in show_cols if c in near.columns]
            display(near.sort_values("criteria_met", ascending=False).head(20)[show_cols])
else:
    print("Load diamonds_full.csv first (cell above).")